# Proyecto 2: retratos profesionales con Stable Diffusion

CS5363 Planificación y Toma de Decisiones en IA, UTEC 2026-2. Grupo 7: José Guillermo Gálvez Pacori, Matías Alonso Meneses Roncal, Alejandro Joel Ore García.

Servidor GPU y experimentos del sistema. Se ejecuta en Colab con GPU T4 usando Entorno de ejecución > Ejecutar todas.

Métodos de generación:

- Nivel intermedio: Stable Diffusion 1.5 con ControlNet Canny, en modo img2img.
- Nivel básico: Stable Diffusion 1.5 en modo img2img, sin ControlNet.

## 1. Dependencias

Se retira `torchao` porque la versión instalada en Colab (0.10) no es compatible con `peft` al cargar el LoRA.

In [ ]:
!pip install -q diffusers transformers accelerate peft safetensors fastapi uvicorn
!pip uninstall -y -q torchao

## 2. Módulos del proyecto

Copia de los módulos de `Project-II/src` que usa el servidor.

In [ ]:
import os

PROJECT_DIR = "/content/proyecto2"
os.makedirs(f"{PROJECT_DIR}/src", exist_ok=True)
os.chdir(PROJECT_DIR)
open("src/__init__.py", "w").close()
print("Directorio de trabajo:", os.getcwd())

In [ ]:
%%writefile src/config.py
from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Optional


class SocialPlatform(str, Enum):
    LINKEDIN = "LinkedIn"
    WHATSAPP = "WhatsApp"
    GITHUB = "GitHub / Portafolio"
    CREATIVE = "Redes & Contenido"


class FormalityLevel(str, Enum):
    BLACK_TIE = "Black Tie"
    FORMAL = "Formal"
    BUSINESS_CASUAL = "Business Casual"
    SMART_CASUAL = "Smart Casual"
    TECH_CASUAL = "Tech Casual"
    CREATIVE_STUDIO = "Creativo de Estudio"


@dataclass
class StylePreset:
    id: str
    platform: SocialPlatform
    formality: FormalityLevel
    title: str
    description: str
    default_prompt: str
    recommended_lighting: str
    recommended_background: str
    recommended_expression: str
    recommended_attire: str
    guidance_scale: float = 7.5
    denoising_strength: float = 0.45


@dataclass
class PresetPackage:
    id: str
    name: str
    description: str
    preset_ids: List[str]


STYLE_PRESETS: Dict[str, StylePreset] = {
    "linkedin_black_tie": StylePreset(
        id="linkedin_black_tie",
        platform=SocialPlatform.LINKEDIN,
        formality=FormalityLevel.BLACK_TIE,
        title="LinkedIn - Black Tie",
        description="Máxima elegancia ejecutiva con terno de gala o vestido formal oscuro.",
        default_prompt="fotografía de retrato profesional, terno negro de tres piezas de gala, camisa blanca impecable, corbata de seda negra o pajarita, postura erguida de liderazgo, iluminación suave de estudio, fondo neutro degradado elegante, alta resolución 8k",
        recommended_lighting="Iluminación suave de estudio con luz de relleno sutil",
        recommended_background="Fondo neutro gris carbón con viñeta sutil",
        recommended_expression="Mirada directa segura y expresión sobria de liderazgo",
        recommended_attire="Terno de gala negro o vestido ejecutivo de alta costura",
        guidance_scale=8.0,
        denoising_strength=0.40,
    ),
    "linkedin_formal": StylePreset(
        id="linkedin_formal",
        platform=SocialPlatform.LINKEDIN,
        formality=FormalityLevel.FORMAL,
        title="LinkedIn - Formal Clásico",
        description="Estilo corporativo tradicional con terno oscuro, camisa blanca y corbata clásica.",
        default_prompt="fotografía corporativa profesional, terno azul marino o carbón, camisa blanca abotonada, corbata elegante, postura profesional confiada, iluminación difusa de estudio fotográfico, fondo desenfocado de oficina corporativa moderna, nitidez facial alta",
        recommended_lighting="Luz frontal difusa de tres puntos",
        recommended_background="Oficina corporativa moderna con bokeh suave",
        recommended_expression="Sonrisa sutil y cálida transmitiendo confiabilidad",
        recommended_attire="Terno oscuro (azul marino o gris) y camisa blanca",
        guidance_scale=7.5,
        denoising_strength=0.42,
    ),
    "linkedin_business_casual": StylePreset(
        id="linkedin_business_casual",
        platform=SocialPlatform.LINKEDIN,
        formality=FormalityLevel.BUSINESS_CASUAL,
        title="LinkedIn - Business Casual",
        description="Blazer refinado sin corbata o blusa ejecutiva, equilibrando seriedad y cercanía.",
        default_prompt="retrato profesional contemporáneo, blazer azul marino sin corbata sobre camisa celeste de cuello abierto, postura relajada pero profesional, iluminación natural lateral suave de ventana, fondo de espacio de coworking elegante con profundidad de campo, enfoque nítido",
        recommended_lighting="Luz natural de ventana cálida y envolvente",
        recommended_background="Espacio de trabajo colaborativo o ventanal con luz diurna",
        recommended_expression="Expresión accesible con ligera sonrisa abierta",
        recommended_attire="Blazer moderno estructurado sobre camisa sin corbata",
        guidance_scale=7.5,
        denoising_strength=0.45,
    ),
    "linkedin_smart_casual": StylePreset(
        id="linkedin_smart_casual",
        platform=SocialPlatform.LINKEDIN,
        formality=FormalityLevel.SMART_CASUAL,
        title="LinkedIn - Smart Casual",
        description="Camisa o suéter pulcro con porte distinguido para sectores innovadores.",
        default_prompt="retrato profesional moderno de negocios, camisa de lino de cuello mao o suéter ligero de cuello redondo pulcro, postura cómoda y dinámica, iluminación cálida de estudio ambiental, fondo arquitectónico neutro desenfocado, textura de piel realista",
        recommended_lighting="Iluminación ambiental cálida con contraste moderado",
        recommended_background="Pared arquitectónica de concreto pulido o madera clara desenfocada",
        recommended_expression="Sonrisa genuina y expresión atenta",
        recommended_attire="Camisa sport elegante o suéter de punto fino pulcro",
        guidance_scale=7.0,
        denoising_strength=0.48,
    ),
    "whatsapp_smart_casual": StylePreset(
        id="whatsapp_smart_casual",
        platform=SocialPlatform.WHATSAPP,
        formality=FormalityLevel.SMART_CASUAL,
        title="WhatsApp - Smart Casual Cercano",
        description="Fotografía amigable y nítida para mensajería profesional y networking directo.",
        default_prompt="foto de perfil clara y accesible, polo camisero o blusa casual pulcra, iluminación diurna brillante y limpia, fondo luminoso neutro ligeramente desenfocado, encuadre cerrado centrado en rostro, expresión afable y profesional",
        recommended_lighting="Luz diurna frontal limpia y sin sombras duras",
        recommended_background="Fondo luminoso y limpio con colores claros",
        recommended_expression="Sonrisa amplia y cordial comunicando apertura",
        recommended_attire="Camisa sport clara o polo camisero presentable",
        guidance_scale=7.0,
        denoising_strength=0.45,
    ),
    "whatsapp_business_casual": StylePreset(
        id="whatsapp_business_casual",
        platform=SocialPlatform.WHATSAPP,
        formality=FormalityLevel.BUSINESS_CASUAL,
        title="WhatsApp - Business Casual",
        description="Presencia corporativa ágil para canales de mensajería empresarial y clientes.",
        default_prompt="foto de perfil ejecutiva para mensajería de negocios, blazer casual sobre camiseta lisa pulcra o blusa ejecutiva, iluminación suave de oficina moderna, fondo neutro con desenfoque suave, composición centrada de alta definición",
        recommended_lighting="Luz suave de oficina con balance de blancos neutro",
        recommended_background="Entorno de oficina neutro minimalista",
        recommended_expression="Mirada confiable y media sonrisa atenta",
        recommended_attire="Blazer liviano sobre prenda lisa básica",
        guidance_scale=7.2,
        denoising_strength=0.43,
    ),
    "github_tech_casual": StylePreset(
        id="github_tech_casual",
        platform=SocialPlatform.GITHUB,
        formality=FormalityLevel.TECH_CASUAL,
        title="GitHub - Tech Casual",
        description="Estilo contemporáneo para desarrolladores e ingenieros de software.",
        default_prompt="retrato profesional para perfil tecnológico de software engineering, hoodie limpio de diseño minimalista o camiseta técnica de alta calidad, iluminación tenue de estudio tecnológico con sutil luz de acento fría, fondo moderno con monitores o estación de trabajo difuminada, detalle nítido",
        recommended_lighting="Luz balanceada con un tenue toque de retroiluminación azul o neutra",
        recommended_background="Estación de desarrollo o setup tecnológico con bokeh profundo",
        recommended_expression="Expresión perspicaz, concentrada y amable",
        recommended_attire="Hoodie minimalista monocromático o camiseta ejecutiva tech",
        guidance_scale=7.5,
        denoising_strength=0.46,
    ),
    "content_creator": StylePreset(
        id="content_creator",
        platform=SocialPlatform.CREATIVE,
        formality=FormalityLevel.CREATIVE_STUDIO,
        title="Redes & Contenido - Creativo de Estudio",
        description="Retrato editorial con iluminación estilizada para perfiles de alto impacto visual.",
        default_prompt="retrato fotográfico editorial para creador de contenido y medios digitales, chaqueta de corte moderno o vestimenta estilizada contemporánea, iluminación dramática de estudio con luz de recorte cinematográfica, fondo con gradiente texturizado oscuro, detalle hiperrealista",
        recommended_lighting="Iluminación cinematográfica de recorte y contraste selectivo",
        recommended_background="Fondo de estudio texturizado con luz de recorte estilizada",
        recommended_expression="Mirada magnética y carismática",
        recommended_attire="Chaqueta moderna o atuendo creativo contemporáneo",
        guidance_scale=8.0,
        denoising_strength=0.50,
    ),
}


PRESET_PACKAGES: Dict[str, PresetPackage] = {
    "package_corporate_mix": PresetPackage(
        id="package_corporate_mix",
        name="Paquete Corporativo Multiplataforma (5 Variaciones)",
        description="Cubre los principales niveles formales para LinkedIn y mensajería profesional ejecutiva.",
        preset_ids=[
            "linkedin_formal",
            "linkedin_business_casual",
            "linkedin_smart_casual",
            "whatsapp_business_casual",
            "whatsapp_smart_casual",
        ],
    ),
    "package_tech_professional": PresetPackage(
        id="package_tech_professional",
        name="Paquete Tech & Networking (5 Variaciones)",
        description="Optimizado para ingenieros, desarrolladores y líderes de tecnología en LinkedIn, GitHub y mensajería.",
        preset_ids=[
            "github_tech_casual",
            "linkedin_business_casual",
            "linkedin_smart_casual",
            "whatsapp_smart_casual",
            "content_creator",
        ],
    ),
    "package_executive_full": PresetPackage(
        id="package_executive_full",
        name="Paquete Ejecutivo Integral (7 Variaciones)",
        description="Conjunto completo de alta formalidad que cubre desde Black Tie hasta Smart Casual y perfiles tech.",
        preset_ids=[
            "linkedin_black_tie",
            "linkedin_formal",
            "linkedin_business_casual",
            "linkedin_smart_casual",
            "whatsapp_business_casual",
            "whatsapp_smart_casual",
            "github_tech_casual",
        ],
    ),
}


NEGATIVE_PROMPT: str = (
    "desnudez, nsfw, sin ropa, ropa reveladora, deformidad facial, ojos asimétricos, "
    "artefactos digitales, baja resolución, sobreexposición, subexposición, rostro borroso, "
    "anatomía incorrecta, dedos adicionales, textura plástica artificial, marcas de agua, texto"
)


def get_preset(preset_id: str) -> Optional[StylePreset]:
    return STYLE_PRESETS.get(preset_id)


def get_all_presets() -> List[StylePreset]:
    return list(STYLE_PRESETS.values())


def get_package(package_id: str) -> Optional[PresetPackage]:
    return PRESET_PACKAGES.get(package_id)


def get_all_packages() -> List[PresetPackage]:
    return list(PRESET_PACKAGES.values())


def build_composite_prompt(preset_id: str, custom_prompt: Optional[str] = None) -> str:
    preset = get_preset(preset_id)
    if not preset:
        base_prompt = "retrato fotográfico profesional de negocios de alta calidad"
    else:
        base_prompt = preset.default_prompt

    if custom_prompt and custom_prompt.strip():
        sanitized_custom = custom_prompt.strip()
        return f"{base_prompt}, personalización solicitada: {sanitized_custom}"

    return base_prompt

In [ ]:
%%writefile src/safety.py
import re
import unicodedata
from dataclasses import dataclass
from typing import List, Optional, Pattern, Tuple


@dataclass
class SafetyCheckResult:
    is_safe: bool
    violation_category: Optional[str] = None
    violation_term: Optional[str] = None
    warning_message: Optional[str] = None


PROHIBITED_TERMS: List[Tuple[str, str]] = [
    ("sin ropa", "desnudez"),
    ("desnudo", "desnudez"),
    ("desnuda", "desnudez"),
    ("desnudez", "desnudez"),
    ("desvestido", "desnudez"),
    ("desvestida", "desnudez"),
    ("en toalla", "ropa inapropiada"),
    ("en ropa interior", "ropa inapropiada"),
    ("ropa interior", "ropa inapropiada"),
    ("lenceria", "ropa inapropiada"),
    ("lencería", "ropa inapropiada"),
    ("bikini", "ropa inapropiada"),
    ("traje de bano", "ropa inapropiada"),
    ("traje de baño", "ropa inapropiada"),
    ("topless", "desnudez"),
    ("shirtless", "desnudez"),
    ("sin camisa", "desnudez"),
    ("sin polo", "desnudez"),
    ("semidesnudo", "desnudez"),
    ("semidesnuda", "desnudez"),
    ("semi desnudo", "desnudez"),
    ("semi desnuda", "desnudez"),
    ("desnudar", "desnudez"),
    ("calzoncillos", "ropa inapropiada"),
    ("underwear", "ropa inapropiada"),
    ("lingerie", "ropa inapropiada"),
    ("nudity", "desnudez"),
    ("nude", "desnudez"),
    ("naked", "desnudez"),
    ("undressed", "desnudez"),
    ("nsfw", "contenido explícito"),
    ("erotico", "contenido explícito"),
    ("erótico", "contenido explícito"),
    ("sexual", "contenido explícito"),
    ("porn", "contenido explícito"),
    ("porno", "contenido explícito"),
    ("sensual explicito", "contenido explícito"),
    ("violencia", "violencia"),
    ("arma", "violencia"),
    ("pistola", "violencia"),
    ("cuchillo", "violencia"),
    ("sangre", "violencia"),
    ("drogas", "sustancias ilícitas"),
    ("droga", "sustancias ilícitas"),
    ("odio", "discurso de odio"),
    ("racismo", "discurso de odio"),
]

PROHIBITED_PATTERNS: List[Tuple[Pattern[str], str]] = [
    (re.compile(r"\bsin\s+(ropa|prendas?|vestir)\b", re.IGNORECASE), "desnudez"),
    (re.compile(r"\bquitar?(le)?\s+(la\s+)?ropa\b", re.IGNORECASE), "desnudez"),
    (re.compile(r"\bno\s+clothes\b", re.IGNORECASE), "desnudez"),
    (re.compile(r"\bwithout\s+clothes\b", re.IGNORECASE), "desnudez"),
    (re.compile(r"\btake\s+off\s+clothes\b", re.IGNORECASE), "desnudez"),
]


LEET_TRANSLATION = str.maketrans({"0": "o", "1": "i", "3": "e", "4": "a", "5": "s", "7": "t", "@": "a", "$": "s"})


def normalize_text(text: str) -> str:
    normalized = unicodedata.normalize("NFKD", text)
    cleaned = "".join(c for c in normalized if not unicodedata.combining(c))
    return cleaned.lower()


def check_prompt_safety(prompt: Optional[str]) -> SafetyCheckResult:
    if not prompt or not prompt.strip():
        return SafetyCheckResult(is_safe=True)

    normalized = normalize_text(prompt)
    for candidate in dict.fromkeys((normalized, normalized.translate(LEET_TRANSLATION))):
        result = _check_normalized(candidate)
        if not result.is_safe:
            return result
    return SafetyCheckResult(is_safe=True)


def _check_normalized(normalized_prompt: str) -> SafetyCheckResult:

    for pattern, category in PROHIBITED_PATTERNS:
        match = pattern.search(normalized_prompt)
        if match:
            return SafetyCheckResult(
                is_safe=False,
                violation_category=category,
                violation_term=match.group(0),
                warning_message=(
                    f"La solicitud fue bloqueada por infringir las directrices éticas del sistema "
                    f"(Categoría: {category.capitalize()}). No se permite la generación o modificación "
                    f"de imágenes profesionales con términos alusivos a '{match.group(0)}'."
                ),
            )

    for term, category in PROHIBITED_TERMS:
        norm_term = normalize_text(term)
        pattern = rf"\b{re.escape(norm_term)}\b"
        if re.search(pattern, normalized_prompt):
            return SafetyCheckResult(
                is_safe=False,
                violation_category=category,
                violation_term=term,
                warning_message=(
                    f"La solicitud fue bloqueada por infringir las directrices éticas del sistema "
                    f"(Categoría: {category.capitalize()}). Se detectó el término no profesional: '{term}'."
                ),
            )

    return SafetyCheckResult(is_safe=True)

In [ ]:
%%writefile src/generator.py
import base64
import io
import os
import time
import uuid
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List, Optional, Tuple

import cv2
import numpy as np
import requests
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageOps

from src.config import (
    NEGATIVE_PROMPT,
    StylePreset,
    build_composite_prompt,
    get_package,
    get_preset,
)
from src.safety import SafetyCheckResult, check_prompt_safety


class EngineMode(str, Enum):
    INTERMEDIATE = "Nivel Intermedio (ControlNet Canny + img2img, GPU)"
    BASIC = "Nivel Básico (Stable Diffusion img2img, GPU)"
    LOCAL = "Contingencia local (filtro OpenCV, sin modelo generativo)"


FACE_CASCADE_FILES = ("haarcascade_frontalface_alt2.xml", "haarcascade_frontalface_default.xml")
FACE_DETECTION_SIDE = 640
FACE_CROP_FACTOR = 1.9
FACE_VERTICAL_OFFSET = 0.15
HEAD_TOP_MARGIN = 0.25


REMOTE_METHOD_BY_MODE = {
    EngineMode.INTERMEDIATE: "controlnet_img2img",
    EngineMode.BASIC: "img2img",
}


@dataclass
class GeneratedVariation:
    variation_id: str
    preset_id: str
    title: str
    platform: str
    formality: str
    image: Image.Image
    prompt_used: str
    negative_prompt: str
    engine_mode: str
    guidance_scale: float
    denoising_strength: float
    recommended_lighting: str
    recommended_background: str
    recommended_expression: str
    recommended_attire: str
    created_at: float
    controlnet_edge_preview: Optional[Image.Image] = None


class GenerationEngine:
    def __init__(self, mode: EngineMode = EngineMode.LOCAL):
        self.mode = mode
        self.face_cascades = self._init_face_cascades()
        self.last_face_box: Optional[Tuple[int, int, int, int]] = None
        self.hf_token = os.getenv("HF_TOKEN", "")

    def _init_face_cascades(self) -> List[cv2.CascadeClassifier]:
        cascades = []
        for name in FACE_CASCADE_FILES:
            try:
                path = cv2.data.haarcascades + name
                if os.path.exists(path):
                    cascade = cv2.CascadeClassifier(path)
                    if not cascade.empty():
                        cascades.append(cascade)
            except Exception:
                continue
        return cascades

    def detect_face(self, image_rgb: np.ndarray) -> Optional[Tuple[int, int, int, int]]:
        gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
        scale = min(1.0, FACE_DETECTION_SIDE / max(gray.shape))
        if scale < 1.0:
            gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
        gray = cv2.equalizeHist(gray)

        for cascade in self.face_cascades:
            faces = cascade.detectMultiScale(gray, scaleFactor=1.05, minNeighbors=5, minSize=(40, 40))
            if len(faces) > 0:
                x, y, w, h = max(faces, key=lambda rect: rect[2] * rect[3])
                return tuple(int(round(value / scale)) for value in (x, y, w, h))
        return None

    def preprocess_face(self, pil_image: Image.Image, target_size: Tuple[int, int] = (512, 512)) -> Image.Image:
        image_rgb = np.array(pil_image.convert("RGB"))
        height, width, _ = image_rgb.shape
        face = self.detect_face(image_rgb)
        self.last_face_box = face

        if face is None:
            return ImageOps.fit(pil_image.convert("RGB"), target_size, method=Image.Resampling.LANCZOS)

        x, y, w, h = face
        crop_size = min(int(max(w, h) * FACE_CROP_FACTOR), width, height)
        center_x = x + w // 2
        center_y = y + h // 2 + int(h * FACE_VERTICAL_OFFSET)

        x1 = center_x - crop_size // 2
        y1 = center_y - crop_size // 2
        y1 = min(y1, y - int(h * HEAD_TOP_MARGIN))
        x1 = min(max(0, x1), width - crop_size)
        y1 = min(max(0, y1), height - crop_size)

        cropped = image_rgb[y1:y1 + crop_size, x1:x1 + crop_size]
        interpolation = cv2.INTER_AREA if crop_size > target_size[0] else cv2.INTER_CUBIC
        resized = cv2.resize(cropped, target_size, interpolation=interpolation)
        return Image.fromarray(resized)

    def extract_controlnet_edges(self, pil_image: Image.Image, low_threshold: int = 100, high_threshold: int = 200) -> Image.Image:
        image_np = np.array(pil_image.convert("RGB"))
        gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
        blurred = cv2.GaussianBlur(gray, (3, 3), 0)
        edges = cv2.Canny(blurred, low_threshold, high_threshold)
        return Image.fromarray(edges)

    def _apply_portrait_depth_blur(self, pil_img: Image.Image) -> Image.Image:
        img_np = np.array(pil_img.convert("RGB"))
        h, w, _ = img_np.shape

        mask = np.zeros((h, w), dtype=np.uint8)
        cx, cy = w // 2, int(h * 0.42)
        rx, ry = int(w * 0.28), int(h * 0.28)
        cv2.ellipse(mask, (cx, cy), (rx, ry), 0, 0, 360, 255, -1)
        cv2.ellipse(mask, (cx, int(h * 0.88)), (int(w * 0.45), int(h * 0.32)), 0, 0, 360, 255, -1)

        mask_blurred = cv2.GaussianBlur(mask, (65, 65), 0).astype(np.float32) / 255.0
        mask_3ch = np.repeat(mask_blurred[:, :, np.newaxis], 3, axis=2)

        bg_blurred = cv2.GaussianBlur(img_np, (51, 51), 0)
        composite = (img_np * mask_3ch + bg_blurred * (1.0 - mask_3ch)).astype(np.uint8)
        return Image.fromarray(composite)

    def _apply_preset_styling(
        self,
        base_img: Image.Image,
        preset: StylePreset,
    ) -> Image.Image:
        depth_img = self._apply_portrait_depth_blur(base_img)
        img = depth_img.copy()
        w, h = img.size

        if preset.id == "linkedin_black_tie":
            contrast = ImageEnhance.Contrast(img).enhance(1.25)
            sharp = ImageEnhance.Sharpness(contrast).enhance(1.3)
            overlay = Image.new("RGBA", (w, h), (0, 0, 0, 0))
            draw = ImageDraw.Draw(overlay)
            draw.rectangle([0, int(h * 0.65), w, h], fill=(10, 12, 16, 175))
            draw.polygon([(int(w * 0.32), int(h * 0.65)), (int(w * 0.50), int(h * 0.85)), (int(w * 0.68), int(h * 0.65))], fill=(245, 245, 250, 190))
            draw.polygon([(int(w * 0.46), int(h * 0.66)), (int(w * 0.50), int(h * 0.84)), (int(w * 0.54), int(h * 0.66))], fill=(15, 15, 20, 230))
            draw.line([(0, int(h * 0.65)), (w, int(h * 0.65))], fill=(255, 255, 255, 60), width=2)
            img = Image.alpha_composite(sharp.convert("RGBA"), overlay).convert("RGB")

        elif preset.id == "linkedin_formal":
            contrast = ImageEnhance.Contrast(img).enhance(1.18)
            sharp = ImageEnhance.Sharpness(contrast).enhance(1.25)
            r, g, b = sharp.split()
            b = b.point(lambda i: min(255, int(i * 1.06)))
            r = r.point(lambda i: int(i * 0.97))
            tinted = Image.merge("RGB", (r, g, b))
            overlay = Image.new("RGBA", (w, h), (0, 0, 0, 0))
            draw = ImageDraw.Draw(overlay)
            draw.rectangle([0, int(h * 0.67), w, h], fill=(16, 26, 48, 160))
            draw.polygon([(int(w * 0.34), int(h * 0.67)), (int(w * 0.50), int(h * 0.84)), (int(w * 0.66), int(h * 0.67))], fill=(240, 245, 255, 185))
            draw.polygon([(int(w * 0.47), int(h * 0.68)), (int(w * 0.50), int(h * 0.83)), (int(w * 0.53), int(h * 0.68))], fill=(18, 55, 115, 220))
            draw.line([(0, int(h * 0.67)), (w, int(h * 0.67))], fill=(56, 189, 248, 70), width=2)
            img = Image.alpha_composite(tinted.convert("RGBA"), overlay).convert("RGB")

        elif preset.id == "linkedin_business_casual":
            r, g, b = img.split()
            r = r.point(lambda i: min(255, int(i * 1.07)))
            b = b.point(lambda i: int(i * 0.93))
            warm = Image.merge("RGB", (r, g, b))
            bright = ImageEnhance.Brightness(warm).enhance(1.05)
            contrast = ImageEnhance.Contrast(bright).enhance(1.12)
            overlay = Image.new("RGBA", (w, h), (0, 0, 0, 0))
            draw = ImageDraw.Draw(overlay)
            draw.rectangle([0, int(h * 0.69), w, h], fill=(28, 42, 60, 135))
            draw.polygon([(int(w * 0.36), int(h * 0.69)), (int(w * 0.50), int(h * 0.82)), (int(w * 0.64), int(h * 0.69))], fill=(225, 235, 250, 170))
            img = Image.alpha_composite(contrast.convert("RGBA"), overlay).convert("RGB")

        elif preset.id == "linkedin_smart_casual":
            r, g, b = img.split()
            r = r.point(lambda i: min(255, int(i * 1.05)))
            g = g.point(lambda i: min(255, int(i * 1.02)))
            warm = Image.merge("RGB", (r, g, b))
            bright = ImageEnhance.Brightness(warm).enhance(1.04)
            contrast = ImageEnhance.Contrast(bright).enhance(1.10)
            overlay = Image.new("RGBA", (w, h), (0, 0, 0, 0))
            draw = ImageDraw.Draw(overlay)
            draw.rectangle([0, int(h * 0.71), w, h], fill=(42, 38, 34, 115))
            img = Image.alpha_composite(contrast.convert("RGBA"), overlay).convert("RGB")

        elif preset.id == "whatsapp_smart_casual":
            bright = ImageEnhance.Brightness(img).enhance(1.08)
            color = ImageEnhance.Color(bright).enhance(1.10)
            contrast = ImageEnhance.Contrast(color).enhance(1.07)
            img = contrast

        elif preset.id == "whatsapp_business_casual":
            bright = ImageEnhance.Brightness(img).enhance(1.04)
            contrast = ImageEnhance.Contrast(bright).enhance(1.13)
            overlay = Image.new("RGBA", (w, h), (0, 0, 0, 0))
            draw = ImageDraw.Draw(overlay)
            draw.rectangle([0, int(h * 0.70), w, h], fill=(32, 38, 48, 120))
            img = Image.alpha_composite(contrast.convert("RGBA"), overlay).convert("RGB")

        elif preset.id == "github_tech_casual":
            r, g, b = img.split()
            b = b.point(lambda i: min(255, int(i * 1.14)))
            r = r.point(lambda i: int(i * 0.93))
            cool = Image.merge("RGB", (r, g, b))
            contrast = ImageEnhance.Contrast(cool).enhance(1.18)
            overlay = Image.new("RGBA", (w, h), (0, 0, 0, 0))
            draw = ImageDraw.Draw(overlay)
            draw.rectangle([0, int(h * 0.68), w, h], fill=(15, 23, 42, 160))
            draw.line([(0, int(h * 0.68)), (w, int(h * 0.68))], fill=(56, 189, 248, 85), width=2)
            img = Image.alpha_composite(contrast.convert("RGBA"), overlay).convert("RGB")

        elif preset.id == "content_creator":
            contrast = ImageEnhance.Contrast(img).enhance(1.30)
            sharp = ImageEnhance.Sharpness(contrast).enhance(1.35)
            color = ImageEnhance.Color(sharp).enhance(1.15)
            overlay = Image.new("RGBA", (w, h), (0, 0, 0, 0))
            draw = ImageDraw.Draw(overlay)
            draw.rectangle([0, int(h * 0.66), w, h], fill=(22, 12, 32, 145))
            img = Image.alpha_composite(color.convert("RGBA"), overlay).convert("RGB")

        else:
            img = ImageEnhance.Contrast(img).enhance(1.10)

        return img

    def _synthesize_variation(
        self,
        preprocessed_img: Image.Image,
        preset: StylePreset,
        custom_prompt: Optional[str],
    ) -> GeneratedVariation:
        full_prompt = build_composite_prompt(preset.id, custom_prompt)
        canny_edges = self.extract_controlnet_edges(preprocessed_img)
        transformed = self._apply_preset_styling(preprocessed_img, preset)

        unique_suffix = uuid.uuid4().hex[:6]
        variation_id = f"var_{preset.id}_{unique_suffix}"

        return GeneratedVariation(
            variation_id=variation_id,
            preset_id=preset.id,
            title=preset.title,
            platform=preset.platform.value,
            formality=preset.formality.value,
            image=transformed,
            prompt_used=full_prompt,
            negative_prompt=NEGATIVE_PROMPT,
            engine_mode=EngineMode.LOCAL.value,
            guidance_scale=preset.guidance_scale,
            denoising_strength=preset.denoising_strength,
            recommended_lighting=preset.recommended_lighting,
            recommended_background=preset.recommended_background,
            recommended_expression=preset.recommended_expression,
            recommended_attire=preset.recommended_attire,
            created_at=time.time(),
            controlnet_edge_preview=canny_edges,
        )

    def generate_batch(
        self,
        input_image: Image.Image,
        preset_ids: List[str],
        custom_prompt: Optional[str] = None,
    ) -> Tuple[List[GeneratedVariation], SafetyCheckResult]:
        safety_result = check_prompt_safety(custom_prompt)
        if not safety_result.is_safe:
            return [], safety_result

        preprocessed = self.preprocess_face(input_image)
        variations: List[GeneratedVariation] = []

        for pid in preset_ids:
            preset = get_preset(pid)
            if not preset:
                continue
            variation = self._synthesize_variation(preprocessed, preset, custom_prompt)
            variations.append(variation)

        return variations, safety_result

    def generate_via_remote_gpu(
        self,
        input_image: Image.Image,
        preset_ids: List[str],
        backend_url: str,
        custom_prompt: Optional[str] = None,
        mode: EngineMode = EngineMode.INTERMEDIATE,
        num_steps: int = 25,
    ) -> Tuple[List[GeneratedVariation], SafetyCheckResult]:
        safety_result = check_prompt_safety(custom_prompt)
        if not safety_result.is_safe:
            return [], safety_result

        preprocessed = self.preprocess_face(input_image)
        buffered = io.BytesIO()
        preprocessed.save(buffered, format="PNG")
        b64_image = base64.b64encode(buffered.getvalue()).decode("utf-8")

        clean_url = backend_url.rstrip("/")
        endpoint = f"{clean_url}/generate"
        payload = {
            "image_base64": b64_image,
            "preset_ids": preset_ids,
            "custom_prompt": custom_prompt,
            "method": REMOTE_METHOD_BY_MODE.get(mode, "controlnet_img2img"),
            "num_steps": num_steps,
        }

        try:
            try:
                response = requests.post(endpoint, json=payload, timeout=300)
            except requests.ConnectionError:
                time.sleep(2)
                response = requests.post(endpoint, json=payload, timeout=300)
            if response.status_code == 200:
                data = response.json()
                variations: List[GeneratedVariation] = []
                for var_data in data.get("variations", []):
                    img_b64 = var_data["image_base64"]
                    if "," in img_b64:
                        img_b64 = img_b64.split(",", 1)[1]
                    img_bytes = base64.b64decode(img_b64)
                    pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
                    variations.append(
                        GeneratedVariation(
                            variation_id=var_data["variation_id"],
                            preset_id=var_data["preset_id"],
                            title=var_data["title"],
                            platform=var_data["platform"],
                            formality=var_data["formality"],
                            image=pil_img,
                            prompt_used=var_data["prompt_used"],
                            negative_prompt=var_data["negative_prompt"],
                            engine_mode=mode.value,
                            guidance_scale=var_data["guidance_scale"],
                            denoising_strength=var_data["denoising_strength"],
                            recommended_lighting=var_data["recommended_lighting"],
                            recommended_background=var_data["recommended_background"],
                            recommended_expression=var_data["recommended_expression"],
                            recommended_attire=var_data["recommended_attire"],
                            created_at=var_data["created_at"],
                            controlnet_edge_preview=self.extract_controlnet_edges(preprocessed),
                        )
                    )
                return variations, safety_result
            else:
                try:
                    detail = response.json().get("detail", "")
                except Exception:
                    detail = ""
                return [], SafetyCheckResult(
                    is_safe=False,
                    warning_message=f"El servidor GPU respondió con código {response.status_code}. {detail}".strip(),
                )
        except Exception as exc:
            return [], SafetyCheckResult(is_safe=False, warning_message=f"Error al conectar con servidor GPU ({str(exc)}).")

    def generate_from_package(
        self,
        input_image: Image.Image,
        package_id: str,
        custom_prompt: Optional[str] = None,
    ) -> Tuple[List[GeneratedVariation], SafetyCheckResult]:
        package = get_package(package_id)
        if not package:
            preset_ids = ["linkedin_formal", "linkedin_business_casual", "whatsapp_smart_casual"]
        else:
            preset_ids = package.preset_ids

        return self.generate_batch(input_image, preset_ids, custom_prompt)

In [ ]:
%%writefile src/diffusion.py
import time
from dataclasses import dataclass
from typing import Any, Dict, Optional

import cv2
import numpy as np
from PIL import Image

BASE_MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
CONTROLNET_MODEL_ID = "lllyasviel/control_v11p_sd15_canny"
LORA_REPO_ID = "OedoSoldier/detail-tweaker-lora"
LORA_WEIGHT_NAME = "add_detail.safetensors"
LORA_ADAPTER_NAME = "detail_tweaker"
LORA_SCALE = 0.6

METHOD_CONTROLNET = "controlnet_img2img"
METHOD_BASIC = "img2img"
SUPPORTED_METHODS = (METHOD_CONTROLNET, METHOD_BASIC)

IMAGE_SIZE = 512
CONTROLNET_CONDITIONING_SCALE = 0.85
CANNY_LOW_THRESHOLD = 100
CANNY_HIGH_THRESHOLD = 200
MAX_ERROR_LENGTH = 300

DIFFUSION_QUALITY_SUFFIX = "professional headshot photo, sharp focus on face, realistic skin, 85mm lens"
DIFFUSION_NEGATIVE_PROMPT = (
    "deformed face, distorted eyes, extra fingers, blurry, low quality, cartoon, illustration, painting, "
    "watermark, text, nudity, oversaturated, plastic skin"
)
DEFAULT_DIFFUSION_PROMPT = "person in neat business attire, soft studio lighting, neutral background"
DIFFUSION_PROMPTS: Dict[str, str] = {
    "linkedin_black_tie": "person wearing a black tuxedo, white dress shirt, black bow tie, soft studio lighting, dark gray gradient backdrop",
    "linkedin_formal": "person wearing a navy blue business suit, white shirt, tie, diffused studio lighting, blurred modern corporate office background",
    "linkedin_business_casual": "person wearing a navy blazer over a light blue open collar shirt, no tie, soft window light, blurred coworking space background",
    "linkedin_smart_casual": "person wearing a neat crew neck sweater, warm ambient lighting, blurred light wood architectural background",
    "whatsapp_smart_casual": "person wearing a clean light polo shirt, bright daylight, plain light background, friendly smile, close-up",
    "whatsapp_business_casual": "person wearing a casual blazer over a plain t-shirt, soft office lighting, minimal neutral office background",
    "github_tech_casual": "person wearing a plain dark hoodie, dim lighting with cool blue accent light, blurred desk with computer monitors background",
    "content_creator": "person wearing a modern tailored jacket, dramatic cinematic rim lighting, dark textured studio backdrop",
}


def build_diffusion_prompt(preset_id: str, custom_prompt: Optional[str] = None) -> str:
    parts = []
    if custom_prompt and custom_prompt.strip():
        parts.append(custom_prompt.strip())
    parts.append(DIFFUSION_PROMPTS.get(preset_id, DEFAULT_DIFFUSION_PROMPT))
    parts.append(DIFFUSION_QUALITY_SUFFIX)
    return ", ".join(parts)


def _short_error(exc: Exception) -> str:
    return f"{type(exc).__name__}: {exc}"[:MAX_ERROR_LENGTH]


@dataclass
class DiffusionResult:
    image: Image.Image
    method: str
    strength: float
    guidance_scale: float
    num_steps: int
    seed: int
    lora_active: bool
    nsfw_flagged: bool
    elapsed_seconds: float


def prepare_init_image(pil_image: Image.Image) -> Image.Image:
    return pil_image.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)


def extract_canny_edges(
    pil_image: Image.Image,
    low_threshold: int = CANNY_LOW_THRESHOLD,
    high_threshold: int = CANNY_HIGH_THRESHOLD,
) -> Image.Image:
    image_np = np.array(prepare_init_image(pil_image))
    gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    edges = cv2.Canny(blurred, low_threshold, high_threshold)
    return Image.fromarray(np.stack([edges] * 3, axis=-1))


class DiffusionEngine:
    def __init__(self) -> None:
        self.controlnet_pipe: Any = None
        self.basic_pipe: Any = None
        self.device = "cpu"
        self.device_name = "CPU"
        self.lora_loaded = False
        self.lora_error = ""
        self.load_error = ""

    @property
    def is_loaded(self) -> bool:
        return self.controlnet_pipe is not None and self.basic_pipe is not None

    def load(self, use_lora: bool = True) -> bool:
        if self.is_loaded:
            return True

        try:
            import torch
            from diffusers import (
                ControlNetModel,
                StableDiffusionControlNetImg2ImgPipeline,
                StableDiffusionImg2ImgPipeline,
                UniPCMultistepScheduler,
            )

            if torch.cuda.is_available():
                self.device = "cuda"
                self.device_name = torch.cuda.get_device_name(0)
                dtype = torch.float16
            else:
                self.device = "cpu"
                self.device_name = "CPU"
                dtype = torch.float32

            controlnet = ControlNetModel.from_pretrained(CONTROLNET_MODEL_ID, torch_dtype=dtype)
            controlnet_pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
                BASE_MODEL_ID,
                controlnet=controlnet,
                torch_dtype=dtype,
            )
            controlnet_pipe.scheduler = UniPCMultistepScheduler.from_config(controlnet_pipe.scheduler.config)
            controlnet_pipe = controlnet_pipe.to(self.device)
            controlnet_pipe.set_progress_bar_config(disable=True)

            basic_pipe = StableDiffusionImg2ImgPipeline(
                vae=controlnet_pipe.vae,
                text_encoder=controlnet_pipe.text_encoder,
                tokenizer=controlnet_pipe.tokenizer,
                unet=controlnet_pipe.unet,
                scheduler=UniPCMultistepScheduler.from_config(controlnet_pipe.scheduler.config),
                safety_checker=controlnet_pipe.safety_checker,
                feature_extractor=controlnet_pipe.feature_extractor,
            )
            basic_pipe.set_progress_bar_config(disable=True)

            self.controlnet_pipe = controlnet_pipe
            self.basic_pipe = basic_pipe
            self.load_error = ""
        except Exception as exc:
            self.controlnet_pipe = None
            self.basic_pipe = None
            self.load_error = _short_error(exc)
            return False

        if use_lora:
            self._load_lora()
        return True

    def _load_lora(self) -> None:
        try:
            self.controlnet_pipe.load_lora_weights(
                LORA_REPO_ID,
                weight_name=LORA_WEIGHT_NAME,
                adapter_name=LORA_ADAPTER_NAME,
            )
            self.controlnet_pipe.set_adapters([LORA_ADAPTER_NAME], adapter_weights=[LORA_SCALE])
            self.lora_loaded = True
            self.lora_error = ""
        except Exception as exc:
            self.lora_loaded = False
            self.lora_error = _short_error(exc)
            try:
                self.controlnet_pipe.unload_lora_weights()
            except Exception:
                pass

    def _set_lora_active(self, active: bool) -> bool:
        if not self.lora_loaded:
            return False
        if active:
            self.controlnet_pipe.enable_lora()
        else:
            self.controlnet_pipe.disable_lora()
        return active

    def describe(self) -> Dict[str, Any]:
        return {
            "pipeline_loaded": self.is_loaded,
            "load_error": self.load_error,
            "device": self.device,
            "device_name": self.device_name,
            "base_model": BASE_MODEL_ID,
            "controlnet_model": CONTROLNET_MODEL_ID,
            "lora_repo": LORA_REPO_ID,
            "lora_weight_name": LORA_WEIGHT_NAME,
            "lora_scale": LORA_SCALE,
            "lora_loaded": self.lora_loaded,
            "lora_error": self.lora_error,
            "methods": list(SUPPORTED_METHODS),
        }

    def generate(
        self,
        init_image: Image.Image,
        prompt: str,
        negative_prompt: str,
        method: str,
        strength: float,
        guidance_scale: float,
        num_steps: int,
        seed: int,
        use_lora: bool = True,
        control_image: Optional[Image.Image] = None,
    ) -> DiffusionResult:
        import torch

        if method not in SUPPORTED_METHODS:
            raise ValueError(f"Unsupported generation method: {method}")
        if not self.is_loaded:
            raise RuntimeError(self.load_error or "Diffusion pipelines are not loaded.")

        init = prepare_init_image(init_image)
        generator = torch.Generator(device="cpu").manual_seed(seed)
        lora_active = self._set_lora_active(use_lora)

        start_time = time.time()
        if method == METHOD_CONTROLNET:
            output = self.controlnet_pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                image=init,
                control_image=control_image or extract_canny_edges(init),
                strength=strength,
                guidance_scale=guidance_scale,
                num_inference_steps=num_steps,
                controlnet_conditioning_scale=CONTROLNET_CONDITIONING_SCALE,
                generator=generator,
            )
        else:
            output = self.basic_pipe(
                prompt=prompt,
                negative_prompt=negative_prompt,
                image=init,
                strength=strength,
                guidance_scale=guidance_scale,
                num_inference_steps=num_steps,
                generator=generator,
            )
        elapsed = time.time() - start_time

        flags = getattr(output, "nsfw_content_detected", None) or [False]
        return DiffusionResult(
            image=output.images[0],
            method=method,
            strength=strength,
            guidance_scale=guidance_scale,
            num_steps=num_steps,
            seed=seed,
            lora_active=lora_active,
            nsfw_flagged=bool(flags[0]),
            elapsed_seconds=round(elapsed, 3),
        )

In [ ]:
%%writefile src/gpu_server.py
import base64
import io
import os
import sys
import time
from contextlib import asynccontextmanager
from typing import List, Optional

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from PIL import Image

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from src.config import get_preset
from src.diffusion import (
    DIFFUSION_NEGATIVE_PROMPT,
    METHOD_CONTROLNET,
    SUPPORTED_METHODS,
    DiffusionEngine,
    build_diffusion_prompt,
)
from src.safety import check_prompt_safety

BASE_SEED = 1234

engine = DiffusionEngine()


@asynccontextmanager
async def lifespan(_: FastAPI):
    engine.load()
    yield


app = FastAPI(title="ControlNet Portrait Generation GPU Backend", version="2.0.0", lifespan=lifespan)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


class GenerationRequest(BaseModel):
    image_base64: str
    preset_ids: List[str]
    custom_prompt: Optional[str] = None
    method: str = METHOD_CONTROLNET
    num_steps: int = 25
    guidance_scale: Optional[float] = None
    strength: Optional[float] = None
    seed: Optional[int] = None
    use_lora: bool = True


class VariationResponse(BaseModel):
    variation_id: str
    preset_id: str
    title: str
    platform: str
    formality: str
    image_base64: str
    prompt_used: str
    negative_prompt: str
    method: str
    guidance_scale: float
    denoising_strength: float
    seed: int
    lora_active: bool
    nsfw_flagged: bool
    elapsed_seconds: float
    recommended_lighting: str
    recommended_background: str
    recommended_expression: str
    recommended_attire: str
    created_at: float


class GenerationResponse(BaseModel):
    success: bool
    variations: List[VariationResponse]
    device: str
    method: str
    elapsed_seconds: float
    message: Optional[str] = None


def decode_base64_to_image(b64_string: str) -> Image.Image:
    if "," in b64_string:
        b64_string = b64_string.split(",", 1)[1]
    image_data = base64.b64decode(b64_string)
    return Image.open(io.BytesIO(image_data)).convert("RGB")


def encode_image_to_base64(image: Image.Image) -> str:
    buffered = io.BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")


@app.get("/health")
def health_check():
    status = engine.describe()
    status.update(
        {
            "status": "online",
            "cuda_available": engine.device == "cuda",
            "timestamp": time.time(),
        }
    )
    return status


@app.post("/generate", response_model=GenerationResponse)
def generate_portraits(request: GenerationRequest):
    start_time = time.time()

    safety_result = check_prompt_safety(request.custom_prompt)
    if not safety_result.is_safe:
        raise HTTPException(
            status_code=400,
            detail=safety_result.warning_message or "Prompt violates safety guidelines.",
        )

    if request.method not in SUPPORTED_METHODS:
        raise HTTPException(status_code=400, detail=f"Unsupported method. Use one of: {', '.join(SUPPORTED_METHODS)}.")

    try:
        input_image = decode_base64_to_image(request.image_base64)
    except Exception:
        raise HTTPException(status_code=400, detail="Invalid base64 image data.")

    if not engine.load():
        raise HTTPException(status_code=503, detail=f"Diffusion pipeline unavailable ({engine.load_error}).")

    variations: List[VariationResponse] = []

    for index, pid in enumerate(request.preset_ids):
        preset = get_preset(pid)
        if not preset:
            continue

        full_prompt = build_diffusion_prompt(preset.id, request.custom_prompt)
        seed = (request.seed if request.seed is not None else BASE_SEED) + index

        try:
            result = engine.generate(
                init_image=input_image,
                prompt=full_prompt,
                negative_prompt=DIFFUSION_NEGATIVE_PROMPT,
                method=request.method,
                strength=request.strength if request.strength is not None else preset.denoising_strength,
                guidance_scale=request.guidance_scale if request.guidance_scale is not None else preset.guidance_scale,
                num_steps=request.num_steps,
                seed=seed,
                use_lora=request.use_lora,
            )
        except Exception as exc:
            raise HTTPException(status_code=500, detail=f"Generation failed for preset '{preset.id}' ({type(exc).__name__}: {exc}).")

        variations.append(
            VariationResponse(
                variation_id=f"gpu_var_{preset.id}_{int(time.time() * 1000) % 100000}",
                preset_id=preset.id,
                title=preset.title,
                platform=preset.platform.value,
                formality=preset.formality.value,
                image_base64=encode_image_to_base64(result.image),
                prompt_used=full_prompt,
                negative_prompt=DIFFUSION_NEGATIVE_PROMPT,
                method=result.method,
                guidance_scale=result.guidance_scale,
                denoising_strength=result.strength,
                seed=result.seed,
                lora_active=result.lora_active,
                nsfw_flagged=result.nsfw_flagged,
                elapsed_seconds=result.elapsed_seconds,
                recommended_lighting=preset.recommended_lighting,
                recommended_background=preset.recommended_background,
                recommended_expression=preset.recommended_expression,
                recommended_attire=preset.recommended_attire,
                created_at=time.time(),
            )
        )

    elapsed = time.time() - start_time
    return GenerationResponse(
        success=True,
        variations=variations,
        device=engine.device_name,
        method=request.method,
        elapsed_seconds=round(elapsed, 2),
        message=f"Generated {len(variations)} variations with {request.method} on {engine.device_name}.",
    )


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)

## 3. GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No hay GPU disponible. Seleccione un entorno con GPU T4.")
print("GPU:", torch.cuda.get_device_name(0))

## 4. Imagen de entrada

In [ ]:
import io
import sys

from google.colab import files
from PIL import Image, ImageOps

sys.path.insert(0, os.getcwd())

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No se cargó ninguna imagen.")

source_name = next(iter(uploaded))
source_image = ImageOps.exif_transpose(Image.open(io.BytesIO(uploaded[source_name]))).convert("RGB")

from src.generator import GenerationEngine

init_image = GenerationEngine().preprocess_face(source_image)
display(init_image)

## 5. Modelos

Stable Diffusion 1.5, ControlNet Canny y el LoRA `OedoSoldier/detail-tweaker-lora`.

In [ ]:
from src.diffusion import (
    DIFFUSION_NEGATIVE_PROMPT,
    METHOD_BASIC,
    METHOD_CONTROLNET,
    DiffusionEngine,
    build_diffusion_prompt,
    extract_canny_edges,
)

engine = DiffusionEngine()
if not engine.load():
    raise RuntimeError(f"No se pudieron cargar los modelos: {engine.load_error}")

status = engine.describe()
for key, value in status.items():
    print(f"{key}: {value}")
if not engine.lora_loaded:
    print("LoRA no disponible:", engine.lora_error)

## 6. Experimentos

Con la misma imagen y las mismas semillas:

- 5 variaciones del paquete corporativo con cada método.
- Barrido de `strength` (0.45, 0.60 y 0.75).
- Comparación con y sin LoRA.

In [ ]:
import json
import time

from PIL import ImageDraw

from src.config import get_package, get_preset

PACKAGE_ID = "package_corporate_mix"
NUM_STEPS = 25
BASE_SEED = 1234
SWEEP_STRENGTHS = [0.45, 0.60, 0.75]

OUTPUT_DIR = "/content/resultados_colab"
os.makedirs(OUTPUT_DIR, exist_ok=True)

preset_ids = get_package(PACKAGE_ID).preset_ids
canny_image = extract_canny_edges(init_image)
source_image.save(f"{OUTPUT_DIR}/00_foto_original.png")
init_image.save(f"{OUTPUT_DIR}/01_entrada_512.png")
canny_image.save(f"{OUTPUT_DIR}/02_canny.png")

records = []


def run(preset_id, method, seed, group, filename, strength=None, use_lora=True):
    preset = get_preset(preset_id)
    result = engine.generate(
        init_image=init_image,
        prompt=build_diffusion_prompt(preset.id),
        negative_prompt=DIFFUSION_NEGATIVE_PROMPT,
        method=method,
        strength=strength if strength is not None else preset.denoising_strength,
        guidance_scale=preset.guidance_scale,
        num_steps=NUM_STEPS,
        seed=seed,
        use_lora=use_lora,
        control_image=canny_image,
    )
    os.makedirs(f"{OUTPUT_DIR}/{group}", exist_ok=True)
    result.image.save(f"{OUTPUT_DIR}/{group}/{filename}")
    records.append(
        {
            "group": group,
            "file": f"{group}/{filename}",
            "preset_id": preset.id,
            "title": preset.title,
            "prompt": build_diffusion_prompt(preset.id),
            "method": result.method,
            "strength": result.strength,
            "guidance_scale": result.guidance_scale,
            "num_steps": result.num_steps,
            "seed": result.seed,
            "lora_active": result.lora_active,
            "nsfw_flagged": result.nsfw_flagged,
            "elapsed_seconds": result.elapsed_seconds,
        }
    )
    print(f"{group:28s} {preset.id:28s} strength={result.strength:.2f} lora={result.lora_active} {result.elapsed_seconds:.2f}s")
    return result.image


def contact_sheet(images, labels, path):
    cell, label_height = 384, 28
    sheet = Image.new("RGB", (cell * len(images), cell + label_height), "white")
    draw = ImageDraw.Draw(sheet)
    for position, (image, label) in enumerate(zip(images, labels)):
        sheet.paste(image.resize((cell, cell)), (position * cell, label_height))
        draw.text((position * cell + 6, 8), label, fill="black")
    sheet.save(path)
    return sheet


engine.generate(init_image, "portrait", DIFFUSION_NEGATIVE_PROMPT, METHOD_CONTROLNET, 0.5, 7.5, 4, 0, control_image=canny_image)

sheets = {}
for method, group in ((METHOD_CONTROLNET, "nivel_intermedio_controlnet"), (METHOD_BASIC, "nivel_basico_img2img")):
    images = [
        run(pid, method, BASE_SEED + index, group, f"var{index + 1}_{pid}.png")
        for index, pid in enumerate(preset_ids)
    ]
    sheets[group] = contact_sheet(
        [init_image] + images,
        ["entrada"] + [get_preset(pid).title for pid in preset_ids],
        f"{OUTPUT_DIR}/resumen_{group}.png",
    )

sweep_preset = preset_ids[0]
sweep_images = [
    run(sweep_preset, METHOD_CONTROLNET, BASE_SEED, "barrido_strength", f"strength_{int(value * 100):03d}.png", strength=value)
    for value in SWEEP_STRENGTHS
]
sheets["barrido_strength"] = contact_sheet(
    [init_image] + sweep_images,
    ["entrada"] + [f"strength {value:.2f}" for value in SWEEP_STRENGTHS],
    f"{OUTPUT_DIR}/resumen_barrido_strength.png",
)

lora_images = [
    run(sweep_preset, METHOD_CONTROLNET, BASE_SEED, "comparacion_lora", "sin_lora.png", strength=0.60, use_lora=False),
    run(sweep_preset, METHOD_CONTROLNET, BASE_SEED, "comparacion_lora", "con_lora.png", strength=0.60, use_lora=True),
]
sheets["comparacion_lora"] = contact_sheet(
    [init_image] + lora_images,
    ["entrada", "sin LoRA", "con LoRA"],
    f"{OUTPUT_DIR}/resumen_comparacion_lora.png",
)


def average_latency(group):
    values = [r["elapsed_seconds"] for r in records if r["group"] == group]
    return round(sum(values) / len(values), 3) if values else None


summary = {
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "package_id": PACKAGE_ID,
    "engine": engine.describe(),
    "average_seconds_per_image": {
        "nivel_intermedio_controlnet": average_latency("nivel_intermedio_controlnet"),
        "nivel_basico_img2img": average_latency("nivel_basico_img2img"),
    },
    "records": records,
}
with open(f"{OUTPUT_DIR}/metricas.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print()
print("Latencia media por imagen:", summary["average_seconds_per_image"])

## 7. Resultados

In [ ]:
for name, sheet in sheets.items():
    print(name)
    display(sheet)

## 8. Descarga

In [ ]:
import shutil

zip_path = shutil.make_archive("/content/resultados_colab", "zip", OUTPUT_DIR)
print(zip_path)
files.download(zip_path)

## 9. Servidor para la aplicación

Levanta la API y la publica con un túnel temporal de Cloudflare. La URL `trycloudflare.com` se ingresa en la aplicación Streamlit. La celda permanece en ejecución mientras el servidor está activo.

In [ ]:
import gc
import subprocess

import requests

del engine
gc.collect()
torch.cuda.empty_cache()

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

server_log = open("/content/gpu_server.log", "w")
server_process = subprocess.Popen(
    ["python", "-m", "uvicorn", "src.gpu_server:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

ready = False
for _ in range(150):
    time.sleep(2)
    try:
        health = requests.get("http://localhost:8000/health", timeout=2).json()
        ready = bool(health.get("pipeline_loaded"))
        break
    except Exception:
        continue

if not ready:
    print(open("/content/gpu_server.log").read()[-3000:])
    raise RuntimeError("El servidor no cargó los modelos.")

print("Servidor listo:", health["device_name"], "| LoRA:", health["lora_loaded"])
!cloudflared tunnel --url http://localhost:8000